In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path = '/content/drive/MyDrive/Job_Recommendation_System/huge_job_recommendation_dataset.csv'

In [ ]:
import os
os.path.exists(path)

True

In [ ]:
data = os.path.join('/content/drive/MyDrive/Job_Recommendation_System', 'huge_job_recommendation_dataset.csv')

In [ ]:
df = pd.read_csv(data)

In [ ]:
df.head()

,user_id,job_id,title,description,location,skills_required,education,experience,user_skills,user_location,relevance_score
0,1,4603,Data Analyst,Perform data analyst tasks in a dynamic enviro...,Alexandria,Git;UI/UX Design,Bachelor's in Design,1 years,Cybersecurity;Java,Port Said,0.20
1,2,8122,AI Engineer,Perform ai engineer tasks in a dynamic environ...,Ismailia,NLP;JavaScript,Bachelor's in Statistics,0 years,Communication;Networking;Azure;SQL,Port Said,0.00
2,3,5105,Data Analyst,Perform data analyst tasks in a dynamic enviro...,Mansoura,Java;Teamwork;UI/UX Design;JavaScript,Bachelor's in Computer Science,4 years,AWS;UI/UX Design,Mansoura,0.75
3,4,10419,AI Engineer,Perform ai engineer tasks in a dynamic environ...,Suez,Azure;NLP;Communication;Python,Master's in Data Science,7 years,Docker;React;Azure;Kubernetes,Fayoum,0.35
4,5,2320,AI Engineer,Perform ai engineer tasks in a dynamic environ...,Alexandria,Figma;Docker,PhD in Computer Science,10 years,NLP;UI/UX Design;R,Suez,0.10


In [ ]:
df.isnull().sum()

,0
user_id,0
job_id,0
title,0
description,0
location,0
skills_required,0
education,0
experience,0
user_skills,0
user_location,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 11 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   user_id          1000000 non-null  int64  
 1   job_id           1000000 non-null  int64  
 2   title            1000000 non-null  object 
 3   description      1000000 non-null  object 
 4   location         1000000 non-null  object 
 5   skills_required  1000000 non-null  object 
 6   education        1000000 non-null  object 
 7   experience       1000000 non-null  object 
 8   user_skills      1000000 non-null  object 
 9   user_location    1000000 non-null  object 
 10  relevance_score  1000000 non-null  float64
dtypes: float64(1), int64(2), object(8)
memory usage: 83.9+ MB


In [ ]:
df.sample()

,user_id,job_id,title,description,location,skills_required,education,experience,user_skills,user_location,relevance_score
620323,620324,10397,Software Engineer,Perform software engineer tasks in a dynamic e...,Aswan,Teamwork;Git;R;Communication,Master's in Cybersecurity,3 years,JavaScript;React;SQL;Java;Cybersecurity,Fayoum,0.2


In [ ]:
print("Unique Jobs\n",df['title'].nunique())
df['title'].value_counts()

Unique Jobs
 10


,count
title,
Network Engineer,100468
AI Engineer,100377
Data Scientist,100300
Cloud Architect,100270
Software Engineer,100155
Cybersecurity Analyst,99932
Data Analyst,99910
DevOps Engineer,99881
UI/UX Designer,99358


**Reasoning**:
Import necessary libraries (`string` and `re`), define a text preprocessing function (`clean_and_tokenize`), and apply this function to the specified text columns in the DataFrame `df` to lowercase, remove punctuation, and tokenize the text.



In [ ]:
import string
import re

def clean_and_tokenize(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize the text
    tokens = text.split()
    return tokens

# Define the text columns to preprocess
text_columns = ['title', 'description', 'skills_required', 'education', 'user_skills']

# Apply the function to each specified column
for col in text_columns:
    df[col] = df[col].apply(clean_and_tokenize)

print("Text data in specified columns has been cleaned and tokenized.")
df.head()

Text data in specified columns has been cleaned and tokenized.


,user_id,job_id,title,description,location,skills_required,education,experience,user_skills,user_location,relevance_score
0,1,4603,"[data, analyst]","[perform, data, analyst, tasks, in, a, dynamic...",Alexandria,"[gituiux, design]","[bachelors, in, design]",1 years,[cybersecurityjava],Port Said,0.20
1,2,8122,"[ai, engineer]","[perform, ai, engineer, tasks, in, a, dynamic,...",Ismailia,[nlpjavascript],"[bachelors, in, statistics]",0 years,[communicationnetworkingazuresql],Port Said,0.00
2,3,5105,"[data, analyst]","[perform, data, analyst, tasks, in, a, dynamic...",Mansoura,"[javateamworkuiux, designjavascript]","[bachelors, in, computer, science]",4 years,"[awsuiux, design]",Mansoura,0.75
3,4,10419,"[ai, engineer]","[perform, ai, engineer, tasks, in, a, dynamic,...",Suez,[azurenlpcommunicationpython],"[masters, in, data, science]",7 years,[dockerreactazurekubernetes],Fayoum,0.35
4,5,2320,"[ai, engineer]","[perform, ai, engineer, tasks, in, a, dynamic,...",Alexandria,[figmadocker],"[phd, in, computer, science]",10 years,"[nlpuiux, designr]",Suez,0.10


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Concatenate the preprocessed tokenized lists into new string columns
# For job features: 'title', 'description', 'skills_required', 'education'
df['combined_job_features'] = df['title'].apply(lambda x: ' '.join(x)) + ' ' +
                              df['description'].apply(lambda x: ' '.join(x)) + ' ' +
                              df['skills_required'].apply(lambda x: ' '.join(x)) + ' ' +
                              df['education'].apply(lambda x: ' '.join(x))

# For user features: 'user_skills'
df['combined_user_features'] = df['user_skills'].apply(lambda x: ' '.join(x))

# Initialize TfidfVectorizer objects for jobs and users separately
# This ensures independent vocabulary learning as per the 'fit and transform' instructions for each.
job_vectorizer = TfidfVectorizer()
user_vectorizer = TfidfVectorizer()

# Fit and transform the 'combined_job_features' column
job_tfidf_matrix = job_vectorizer.fit_transform(df['combined_job_features'])

# Fit and transform the 'combined_user_features' column
user_tfidf_matrix = user_vectorizer.fit_transform(df['combined_user_features'])

# Print the shape of the generated TF-IDF matrices
print("Shape of job_tfidf_matrix:", job_tfidf_matrix.shape)
print("Shape of user_tfidf_matrix:", user_tfidf_matrix.shape)

SyntaxError: invalid syntax (981481804.py, line 5)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Concatenate the preprocessed tokenized lists into new string columns
# For job features: 'title', 'description', 'skills_required', 'education'
df['combined_job_features'] = (df['title'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['description'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['skills_required'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['education'].apply(lambda x: ' '.join(x)))

# For user features: 'user_skills'
df['combined_user_features'] = df['user_skills'].apply(lambda x: ' '.join(x))

# Combine all text features to create a unified vocabulary for the TF-IDF vectorizer
all_text_features = pd.concat([df['combined_job_features'], df['combined_user_features']], axis=0)

# Initialize a single TfidfVectorizer
vectorizer = TfidfVectorizer()

# Fit the vectorizer on the combined text features to learn a unified vocabulary
vectorizer.fit(all_text_features)

# Transform the 'combined_job_features' and 'combined_user_features' using the same fitted vectorizer
job_tfidf_matrix = vectorizer.transform(df['combined_job_features'])
user_tfidf_matrix = vectorizer.transform(df['combined_user_features'])

# Print the shape of the generated TF-IDF matrices
print("Shape of job_tfidf_matrix:", job_tfidf_matrix.shape)
print("Shape of user_tfidf_matrix:", user_tfidf_matrix.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between the user_tfidf_matrix and job_tfidf_matrix
cosine_sim_matrix = cosine_similarity(user_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the cosine_sim_matrix to verify its dimensions
print("Shape of cosine_sim_matrix:", cosine_sim_matrix.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between the user_tfidf_matrix and job_tfidf_matrix
cosine_sim_matrix = cosine_similarity(user_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the cosine_sim_matrix to verify its dimensions
print("Shape of cosine_sim_matrix:", cosine_sim_matrix.shape)

## Calculate Job Similarity


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between the job_tfidf_matrix and user_tfidf_matrix
cosine_sim_matrix = cosine_similarity(user_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the cosine_sim_matrix to verify its dimensions
print("Shape of cosine_sim_matrix:", cosine_sim_matrix.shape)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd # Ensure pandas is imported as it's used in combined_job_features

# Concatenate the preprocessed tokenized lists into new string columns
# For job features: 'title', 'description', 'skills_required', 'education'
df['combined_job_features'] = (df['title'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['description'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['skills_required'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['education'].apply(lambda x: ' '.join(x)))

# For user features: 'user_skills'
df['combined_user_features'] = df['user_skills'].apply(lambda x: ' '.join(x))

# Combine all text features to create a unified vocabulary for the TF-IDF vectorizer
all_text_features = pd.concat([df['combined_job_features'], df['combined_user_features']], axis=0)

# Initialize a single TfidfVectorizer
vectorizer = TfidfVectorizer()

# Fit the vectorizer on the combined text features to learn a unified vocabulary
vectorizer.fit(all_text_features)

# Transform the 'combined_job_features' and 'combined_user_features' using the same fitted vectorizer
job_tfidf_matrix = vectorizer.transform(df['combined_job_features'])
user_tfidf_matrix = vectorizer.transform(df['combined_user_features'])

# Print the shape of the generated TF-IDF matrices
print("Shape of job_tfidf_matrix:", job_tfidf_matrix.shape)
print("Shape of user_tfidf_matrix:", user_tfidf_matrix.shape)

# Calculate the cosine similarity between the user_tfidf_matrix and job_tfidf_matrix
cosine_sim_matrix = cosine_similarity(user_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the cosine_sim_matrix to verify its dimensions
print("Shape of cosine_sim_matrix:", cosine_sim_matrix.shape)

In [ ]:
import pandas as pd
import string
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-load the dataset, as 'df' was not defined
data = '/content/drive/MyDrive/Job_Recommendation_System/huge_job_recommendation_dataset.csv'
df = pd.read_csv(data)

# Re-define the clean_and_tokenize function and apply it to text columns
def clean_and_tokenize(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize the text
    tokens = text.split()
    return tokens

# Define the text columns to preprocess
text_columns = ['title', 'description', 'skills_required', 'education', 'user_skills']

# Apply the function to each specified column
for col in text_columns:
    df[col] = df[col].apply(clean_and_tokenize)

# Concatenate the preprocessed tokenized lists into new string columns
# For job features: 'title', 'description', 'skills_required', 'education'
df['combined_job_features'] = (df['title'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['description'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['skills_required'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['education'].apply(lambda x: ' '.join(x)))

# For user features: 'user_skills'
df['combined_user_features'] = df['user_skills'].apply(lambda x: ' '.join(x))

# Combine all text features to create a unified vocabulary for the TF-IDF vectorizer
all_text_features = pd.concat([df['combined_job_features'], df['combined_user_features']], axis=0)

# Initialize a single TfidfVectorizer
vectorizer = TfidfVectorizer()

# Fit the vectorizer on the combined text features to learn a unified vocabulary
vectorizer.fit(all_text_features)

# Transform the 'combined_job_features' and 'combined_user_features' using the same fitted vectorizer
job_tfidf_matrix = vectorizer.transform(df['combined_job_features'])
user_tfidf_matrix = vectorizer.transform(df['combined_user_features'])

# Print the shape of the generated TF-IDF matrices
print("Shape of job_tfidf_matrix:", job_tfidf_matrix.shape)
print("Shape of user_tfidf_matrix:", user_tfidf_matrix.shape)

# Calculate the cosine similarity between the user_tfidf_matrix and job_tfidf_matrix
cosine_sim_matrix = cosine_similarity(user_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the cosine_sim_matrix to verify its dimensions
print("Shape of cosine_sim_matrix:", cosine_sim_matrix.shape)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between job_tfidf_matrix and itself
# This will identify semantically similar jobs
job_sim_matrix = cosine_similarity(job_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the job_sim_matrix to verify its dimensions
print("Shape of job_sim_matrix:", job_sim_matrix.shape)

In [ ]:
import pandas as pd
import string
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-load the dataset to ensure 'df' is defined and its state is consistent
data = '/content/drive/MyDrive/Job_Recommendation_System/huge_job_recommendation_dataset.csv'
df = pd.read_csv(data)

# Re-define the clean_and_tokenize function and apply it to text columns
def clean_and_tokenize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    return tokens

text_columns = ['title', 'description', 'skills_required', 'education', 'user_skills']
for col in text_columns:
    df[col] = df[col].apply(clean_and_tokenize)

# Concatenate the preprocessed tokenized lists into new string columns
df['combined_job_features'] = (df['title'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['description'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['skills_required'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['education'].apply(lambda x: ' '.join(x)))

df['combined_user_features'] = df['user_skills'].apply(lambda x: ' '.join(x))

# Combine all text features to create a unified vocabulary for the TF-IDF vectorizer
all_text_features = pd.concat([df['combined_job_features'], df['combined_user_features']], axis=0)

# Initialize and fit a single TfidfVectorizer
vectorizer = TfidfVectorizer()
vectorizer.fit(all_text_features)

# Transform the 'combined_job_features' to get job_tfidf_matrix
job_tfidf_matrix = vectorizer.transform(df['combined_job_features'])

# Calculate the cosine similarity between job_tfidf_matrix and itself
job_sim_matrix = cosine_similarity(job_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the job_sim_matrix to verify its dimensions
print("Shape of job_tfidf_matrix:", job_tfidf_matrix.shape)
print("Shape of job_sim_matrix:", job_sim_matrix.shape)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import string
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-load the dataset to ensure 'df' is defined and its state is consistent
data = '/content/drive/MyDrive/Job_Recommendation_System/huge_job_recommendation_dataset.csv'
df = pd.read_csv(data)

# Re-define the clean_and_tokenize function and apply it to text columns
def clean_and_tokenize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    return tokens

text_columns = ['title', 'description', 'skills_required', 'education', 'user_skills']
for col in text_columns:
    df[col] = df[col].apply(clean_and_tokenize)

# Concatenate the preprocessed tokenized lists into new string columns
df['combined_job_features'] = (df['title'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['description'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['skills_required'].apply(lambda x: ' '.join(x)) + ' ' +
                               df['education'].apply(lambda x: ' '.join(x)))

df['combined_user_features'] = df['user_skills'].apply(lambda x: ' '.join(x))

# Combine all text features to create a unified vocabulary for the TF-IDF vectorizer
all_text_features = pd.concat([df['combined_job_features'], df['combined_user_features']], axis=0)

# Initialize and fit a single TfidfVectorizer
vectorizer = TfidfVectorizer()
vectorizer.fit(all_text_features)

# Transform the 'combined_job_features' to get job_tfidf_matrix
job_tfidf_matrix = vectorizer.transform(df['combined_job_features'])

# Transform the 'combined_user_features' to get user_tfidf_matrix
user_tfidf_matrix = vectorizer.transform(df['combined_user_features'])

# Calculate the cosine similarity between job_tfidf_matrix and itself
job_sim_matrix = cosine_similarity(job_tfidf_matrix, job_tfidf_matrix)

# Calculate the cosine similarity between the user_tfidf_matrix and job_tfidf_matrix
cosine_sim_matrix = cosine_similarity(user_tfidf_matrix, job_tfidf_matrix)

# Print the shape of the matrices to verify their dimensions
print("Shape of job_tfidf_matrix:", job_tfidf_matrix.shape)
print("Shape of user_tfidf_matrix:", user_tfidf_matrix.shape)
print("Shape of job_sim_matrix:", job_sim_matrix.shape)
print("Shape of cosine_sim_matrix:", cosine_sim_matrix.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def recommend_jobs_for_user(user_id, N=5):
    # Ensure user_id is within valid range (0-indexed)
    if user_id not in df['user_id'].values:
        print(f"User ID {user_id} not found.")
        return []

    # Get the 0-indexed position
    user_idx = df[df['user_id'] == user_id].index[0]

    # Get the similarity scores for this user from the cosine_sim_matrix
    user_similarity_scores = cosine_sim_matrix[user_idx]
    top_job_indices = user_similarity_scores.argsort()[-N:][::-1]

    # Retrieve job_id and title for the top N recommended jobs
    recommended_jobs = []
    for job_idx in top_job_indices:
        job_id = df.iloc[job_idx]['job_id']
        job_title = ' '.join(df.iloc[job_idx]['title']) # Join tokens back to string
        recommended_jobs.append({'job_id': job_id, 'title': job_title})

    return recommended_jobs

print("Recommendation function 'recommend_jobs_for_user' defined.")

Recommendation function 'recommend_jobs_for_user' defined.
